In [1]:
# Wavelet + BEMD-Inspired Enhancement
# Swin Transformer + ConvNeXt + Cross-Attention Fusion
# Grad-CAM++ Explainable AI

!pip -q install tqdm scipy timm PyWavelets scikit-learn seaborn kagglehub

In [2]:
# 2. IMPORTS AND CONFIG

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance

import pywt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from tqdm.auto import tqdm
import scipy.ndimage as ndi


In [ ]:


SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 5

# First stage: train fusion/classification head
WARMUP_EPOCHS = 4

# Second stage: fine-tune pretrained networks
FINETUNE_EPOCHS = 16

# Dynamic Output Directory (Works on Google Colab, Kaggle, Linux, and Windows)
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else "./outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Artifacts will be saved to:", OUTPUT_DIR)

# Safe Worker count (Prevents deadlock / freezing on Windows & Colab)
NUM_WORKERS = 0 if (os.name == 'nt' or not torch.cuda.is_available()) else 2

CLASS_NAMES = ["CC", "EC", "HGSC", "LGSC", "MC"]

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================
# 3. DOWNLOAD DATASET
# =========================
dataset_path = kagglehub.dataset_download(
    "sunilthite/ovarian-cancer-classification-dataset"
)

data_dir = os.path.join(dataset_path, "Train_Images")

print("Dataset path:", dataset_path)
print("Image directory:", data_dir)
print("Exists:", os.path.exists(data_dir))


# =========================
# 4. COLLECT IMAGE PATHS
# =========================
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

image_paths = []
labels = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(data_dir, class_name)

    if not os.path.isdir(class_dir):
        print("WARNING: Missing class folder:", class_dir)
        continue

    for root, _, files in os.walk(class_dir):
        for fname in files:
            if fname.lower().endswith(VALID_EXTENSIONS):
                image_paths.append(os.path.join(root, fname))
                labels.append(class_idx)

image_paths = np.array(image_paths)
labels = np.array(labels)

print("\nTotal images:", len(image_paths))

for i, class_name in enumerate(CLASS_NAMES):
    print(f"{class_name}: {(labels == i).sum()}")


# =========================
# 5. STRATIFIED TRAIN/VALIDATION SPLIT
# =========================
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

print("\nTraining images:", len(train_paths))
print("Validation images:", len(val_paths))

print("\nTraining distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(name, (train_labels == i).sum())

print("\nValidation distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(name, (val_labels == i).sum())


# =========================
# 6. WAVELET + BEMD-INSPIRED ENHANCEMENT
# =========================
# NOTE:
# This implementation uses wavelet decomposition followed by
# local-mean/residual enhancement. It is BEMD-inspired rather
# than a full mathematical BEMD implementation.
#
# If the manuscript requires "true BEMD", replace this function
# with a validated BEMD implementation and keep the rest of the
# network unchanged.

def wavelet_bemd_inspired_enhancement(pil_img):
    img = np.asarray(pil_img.convert("RGB")).astype(np.float32) / 255.0

    enhanced_channels = []

    for c in range(3):
        channel = img[:, :, c]

        # Haar wavelet decomposition
        cA, (cH, cV, cD) = pywt.dwt2(channel, "db1")

        # Reconstruct
        reconstructed = pywt.idwt2(
            (cA, (cH, cV, cD)), "db1"
        )

        reconstructed = reconstructed[:channel.shape[0],
                                       :channel.shape[1]]

        # Local mean
        local_mean = cv2_box_filter(reconstructed, kernel_size=9)

        # Residual component
        imf = reconstructed - local_mean

        # Enhancement
        enhanced = local_mean + 1.20 * imf

        enhanced = np.clip(enhanced, 0, 1)
        enhanced_channels.append(enhanced)

    enhanced = np.stack(enhanced_channels, axis=-1)

    return Image.fromarray(
        (enhanced * 255).astype(np.uint8)
    )


def cv2_box_filter(image, kernel_size=9):
    """
    Lightweight box filtering without requiring OpenCV.
    """
    return ndi.uniform_filter(image, size=kernel_size, mode="reflect")


# =========================
# 7. IMAGE TRANSFORMS
# =========================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tensor_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_tensor_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

geometry_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15)
])

rgb_color_aug = transforms.ColorJitter(
    brightness=0.12,
    contrast=0.12,
    saturation=0.10,
    hue=0.02
)


# =========================
# 8. DATASET CLASS
# =========================
class OvarianDataset(Dataset):

    def __init__(
        self,
        paths,
        labels,
        train=False
    ):
        self.paths = paths
        self.labels = labels
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        path = self.paths[idx]
        label = int(self.labels[idx])

        image = Image.open(path).convert("RGB")

        if self.train:

            # Same geometric transformation for both modalities
            seed = np.random.randint(0, 1_000_000)

            random.seed(seed)
            torch.manual_seed(seed)
            image_geom = geometry_aug(image)

            random.seed(seed)
            torch.manual_seed(seed)
            enhanced_geom = geometry_aug(image)

            # RGB-specific color augmentation
            image_geom = rgb_color_aug(image_geom)

        else:
            image_geom = image
            enhanced_geom = image

        enhanced = wavelet_bemd_inspired_enhancement(
            enhanced_geom
        )

        rgb_tensor = train_tensor_transform(
            image_geom
        ) if self.train else val_tensor_transform(
            image_geom
        )

        enhanced_tensor = train_tensor_transform(
            enhanced
        ) if self.train else val_tensor_transform(
            enhanced
        )

        return (
            rgb_tensor,
            enhanced_tensor,
            torch.tensor(label, dtype=torch.long),
            path
        )


# =========================
# 9. DATALOADERS
# =========================
train_dataset = OvarianDataset(
    train_paths,
    train_labels,
    train=True
)

val_dataset = OvarianDataset(
    val_paths,
    val_labels,
    train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


# =========================
# 10. VISUALIZE ENHANCEMENT
# =========================
sample_path = train_paths[0]

original = Image.open(sample_path).convert("RGB")
enhanced = wavelet_bemd_inspired_enhancement(original)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(enhanced)
plt.title("Wavelet + BEMD-inspired")
plt.axis("off")

plt.tight_layout()
plt.show()


# =========================
# 11. LOAD PRETRAINED SWIN + CONVNEXT
# =========================
# These are pretrained ImageNet/IN22K-initialized timm models.

SWIN_NAME = "swin_base_patch4_window7_224.ms_in22k_ft_in1k"
CONVNEXT_NAME = "convnext_base.fb_in22k_ft_in1k"

swin = timm.create_model(
    SWIN_NAME,
    pretrained=True,
    num_classes=0,
    global_pool=""
)

convnext = timm.create_model(
    CONVNEXT_NAME,
    pretrained=True,
    num_classes=0,
    global_pool=""
)

print("Swin loaded:", SWIN_NAME)
print("ConvNeXt loaded:", CONVNEXT_NAME)


# =========================
# 12. CROSS-ATTENTION FUSION
# =========================
class CrossAttentionFusion(nn.Module):

    def __init__(
        self,
        dim=512,
        num_heads=8,
        dropout=0.10
    ):
        super().__init__()

        self.swin_to_conv = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.conv_to_swin = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_swin = nn.LayerNorm(dim)
        self.norm_conv = nn.LayerNorm(dim)

        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, swin_tokens, conv_tokens):

        swin_attn, _ = self.swin_to_conv(
            query=swin_tokens,
            key=conv_tokens,
            value=conv_tokens
        )

        conv_attn, _ = self.conv_to_swin(
            query=conv_tokens,
            key=swin_tokens,
            value=swin_tokens
        )

        swin_out = self.norm_swin(
            swin_tokens + swin_attn
        )

        conv_out = self.norm_conv(
            conv_tokens + conv_attn
        )

        # Convert both sequences to the same number of tokens
        # using adaptive 1D pooling.
        swin_out = swin_out.transpose(1, 2)
        conv_out = conv_out.transpose(1, 2)

        target_tokens = min(
            swin_out.shape[-1],
            conv_out.shape[-1]
        )

        swin_out = F.adaptive_avg_pool1d(
            swin_out,
            target_tokens
        ).transpose(1, 2)

        conv_out = F.adaptive_avg_pool1d(
            conv_out,
            target_tokens
        ).transpose(1, 2)

        fused = self.fusion(
            torch.cat(
                [swin_out, conv_out],
                dim=-1
            )
        )

        return fused


# =========================
# 13. ATTENTION POOLING
# =========================
class AttentionPooling(nn.Module):

    def __init__(self, dim=512):

        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, x):

        # x: [B, N, D]
        scores = self.score(x)

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            weights * x,
            dim=1
        )

        return pooled, weights


# =========================
# 14. COMPLETE MODEL
# =========================
class WaveletSwinConvNeXt(nn.Module):

    def __init__(
        self,
        num_classes=5,
        feature_dim=512
    ):
        super().__init__()

        self.swin = swin
        self.convnext = convnext

        # Swin Base output dimension
        swin_dim = self.swin.num_features

        # ConvNeXt Base output dimension
        conv_dim = self.convnext.num_features

        self.swin_projection = nn.Linear(
            swin_dim,
            feature_dim
        )

        self.conv_projection = nn.Linear(
            conv_dim,
            feature_dim
        )

        self.cross_attention = CrossAttentionFusion(
            dim=feature_dim,
            num_heads=8
        )

        self.attention_pool = AttentionPooling(
            dim=feature_dim
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                feature_dim,
                512
            ),

            nn.GELU(),

            nn.BatchNorm1d(512),

            nn.Dropout(0.40),

            nn.Linear(
                512,
                256
            ),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(
        self,
        rgb,
        enhanced,
        return_features=False
    ):

        # ---------------------
        # Swin branch (VRAM-optimized for frozen warmup)
        # ---------------------
        is_swin_grad = any(p.requires_grad for p in self.swin.parameters()) and self.training
        with torch.set_grad_enabled(is_swin_grad):
            swin_features = self.swin.forward_features(
                rgb
            )

        # Swin typically returns [B,H,W,C]
        if swin_features.ndim == 4:

            B, H, W, C = swin_features.shape

            swin_tokens = swin_features.reshape(
                B,
                H * W,
                C
            )

        else:
            swin_tokens = swin_features

        swin_tokens = self.swin_projection(
            swin_tokens
        )

        # ---------------------
        # ConvNeXt branch (VRAM-optimized for frozen warmup)
        # ---------------------
        is_conv_grad = any(p.requires_grad for p in self.convnext.parameters()) and self.training
        with torch.set_grad_enabled(is_conv_grad):
            conv_features = self.convnext.forward_features(
                enhanced
            )

        # ConvNeXt normally returns [B,C,H,W]
        if conv_features.ndim == 4:

            B, C, H, W = conv_features.shape

            conv_tokens = conv_features.flatten(
                2
            ).transpose(
                1, 2
            )

        else:
            conv_tokens = conv_features

        conv_tokens = self.conv_projection(
            conv_tokens
        )

        # ---------------------
        # Cross-attention
        # ---------------------
        fused_tokens = self.cross_attention(
            swin_tokens,
            conv_tokens
        )

        # ---------------------
        # Attention pooling
        # ---------------------
        pooled, attention_weights = self.attention_pool(
            fused_tokens
        )

        logits = self.classifier(
            pooled
        )

        if return_features:

            return {
                "logits": logits,
                "swin_tokens": swin_tokens,
                "conv_tokens": conv_tokens,
                "fused_tokens": fused_tokens,
                "pooled": pooled,
                "attention_weights": attention_weights
            }

        return logits


# =========================
# 15. CREATE MODEL
# =========================
model = WaveletSwinConvNeXt(
    num_classes=NUM_CLASSES
).to(DEVICE)

print(model)


# =========================
# 16. TEST FORWARD PASS
# =========================
model.eval()

with torch.no_grad():

    rgb, enhanced, labels_batch, _ = next(
        iter(train_loader)
    )

    rgb = rgb.to(DEVICE)
    enhanced = enhanced.to(DEVICE)

    output = model(
        rgb,
        enhanced
    )

print("Output shape:", output.shape)

assert output.shape[1] == NUM_CLASSES

print("Forward pass successful.")


# =========================
# 17. CLASS WEIGHTS
# =========================
train_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES
)

class_weights = (
    len(train_labels)
    /
    (NUM_CLASSES * train_counts)
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(DEVICE)

print("Class weights:")
for name, weight in zip(
    CLASS_NAMES,
    class_weights.cpu().numpy()
):
    print(name, round(float(weight), 4))


# =========================
# 18. LOSS + OPTIMIZER
# =========================
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)


# =========================
# 19. FREEZE BACKBONES FOR WARMUP
# =========================
for param in model.swin.parameters():
    param.requires_grad = False

for param in model.convnext.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=WARMUP_EPOCHS
)


# =========================
# 20. TRAINING FUNCTION
# =========================
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training  ", leave=False)
    for rgb, enhanced, targets, _ in pbar:

        rgb = rgb.to(DEVICE, non_blocking=True)
        enhanced = enhanced.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            rgb,
            enhanced
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += (
            loss.item() * targets.size(0)
        )
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{(correct/max(1, total))*100:.1f}%"})

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    return (
        running_loss / total,
        correct / total
    )


# =========================
# 21. VALIDATION FUNCTION
# =========================
@torch.no_grad()
def validate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0
    all_targets = []
    all_predictions = []
    all_probabilities = []

    pbar = tqdm(loader, desc="Validating", leave=False)
    for rgb, enhanced, targets, _ in pbar:

        rgb = rgb.to(DEVICE, non_blocking=True)
        enhanced = enhanced.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        logits = model(
            rgb,
            enhanced
        )

        loss = criterion(
            logits,
            targets
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = logits.argmax(
            dim=1
        )

        running_loss += (
            loss.item() * targets.size(0)
        )

        all_targets.extend(
            targets.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    all_targets = np.array(all_targets)
    all_predictions = np.array(all_predictions)
    all_probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    return (
        running_loss / len(loader.dataset),
        accuracy,
        all_targets,
        all_predictions,
        all_probabilities
    )


# =========================
# 22. TRAINING LOOP
# =========================
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0

print("\n========== STAGE 1: WARMUP ==========")

for epoch in range(WARMUP_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc, _, _, _ = validate(
        model,
        val_loader,
        criterion
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch+1}/{WARMUP_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc*100:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            os.path.join(OUTPUT_DIR, "best_ovarian_model.pth")
        )


# =========================
# 23. UNFREEZE LAST STAGES
# =========================
for param in model.swin.parameters():
    param.requires_grad = True

for param in model.convnext.parameters():
    param.requires_grad = True

# Lower learning rate for pretrained fine-tuning
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=FINETUNE_EPOCHS
)

print("\n========== STAGE 2: FINE-TUNING ==========")

for epoch in range(FINETUNE_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc, _, _, _ = validate(
        model,
        val_loader,
        criterion
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Fine-tune Epoch "
        f"[{epoch+1}/{FINETUNE_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc*100:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            os.path.join(OUTPUT_DIR, "best_ovarian_model.pth")
        )

print(
    f"\nBest validation accuracy: "
    f"{best_val_acc*100:.2f}%"
)


# =========================
# 24. LOAD BEST MODEL
# =========================
model.load_state_dict(
    torch.load(
        os.path.join(OUTPUT_DIR, "best_ovarian_model.pth"),
        map_location=DEVICE
    )
)

model.eval()


# =========================
# 25. TRAINING CURVES
# =========================
epochs_total = len(history["train_loss"])

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(
    range(1, epochs_total + 1),
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    range(1, epochs_total + 1),
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)

plt.plot(
    range(1, epochs_total + 1),
    np.array(history["train_acc"]) * 100,
    label="Training Accuracy"
)

plt.plot(
    range(1, epochs_total + 1),
    np.array(history["val_acc"]) * 100,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# =========================
# 26. FINAL EVALUATION
# =========================
val_loss, val_accuracy, y_true, y_pred, y_prob = validate(
    model,
    val_loader,
    criterion
)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

mcc = matthews_corrcoef(
    y_true,
    y_pred
)

try:
    roc_auc = roc_auc_score(
        y_true,
        y_prob,
        multi_class="ovr",
        average="weighted"
    )
except Exception:
    roc_auc = float("nan")

print("\n================ FINAL RESULTS ================")
print(f"Accuracy          : {val_accuracy*100:.2f}%")
print(f"Weighted Precision : {precision*100:.2f}%")
print(f"Weighted Recall    : {recall*100:.2f}%")
print(f"Weighted F1        : {f1*100:.2f}%")
print(f"Macro F1          : {macro_f1*100:.2f}%")
print(f"Balanced Accuracy  : {balanced_acc*100:.2f}%")
print(f"MCC                : {mcc:.4f}")
print(f"Weighted ROC-AUC   : {roc_auc:.4f}")


# =========================
# 27. CLASSIFICATION REPORT
# =========================
report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

print("\nClassification Report:\n")
print(report)


# =========================
# 28. CONFUSION MATRIX
# =========================
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


# =========================
# 29. CLASS-WISE METRICS
# =========================
class_precision = precision_score(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

class_recall = recall_score(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

class_f1 = f1_score(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

specificity = []

for i in range(NUM_CLASSES):

    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    spec = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    specificity.append(spec)

class_metrics = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": class_precision,
    "Recall/Sensitivity": class_recall,
    "Specificity": specificity,
    "F1-Score": class_f1
})

print(class_metrics)


# =========================
# 30. ROC CURVES
# =========================
y_true_bin = label_binarize(
    y_true,
    classes=np.arange(NUM_CLASSES)
)

plt.figure(figsize=(8, 7))

for i, class_name in enumerate(CLASS_NAMES):

    fpr, tpr, _ = roc_curve(
        y_true_bin[:, i],
        y_prob[:, i]
    )

    class_auc = auc(
        fpr,
        tpr
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC={class_auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# =========================
# 31. GRAD-CAM++ IMPLEMENTATION
# =========================
class GradCAMPlusPlus:

    def __init__(
        self,
        model,
        target_layer
    ):

        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(
            self._forward_hook
        )

        self.backward_handle = target_layer.register_full_backward_hook(
            self._backward_hook
        )

    def _forward_hook(
        self,
        module,
        inputs,
        output
    ):

        self.activations = output

    def _backward_hook(
        self,
        module,
        grad_input,
        grad_output
    ):

        self.gradients = grad_output[0]

    def remove_hooks(self):

        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(
        self,
        rgb,
        enhanced,
        target_class=None
    ):

        self.model.zero_grad(set_to_none=True)

        output = self.model(
            rgb,
            enhanced
        )

        if target_class is None:
            target_class = output.argmax(
                dim=1
            ).item()

        score = output[:, target_class].sum()

        score.backward(
            retain_graph=True
        )

        activations = self.activations
        gradients = self.gradients

        # Handle ConvNeXt spatial feature tensor [B,C,H,W]
        if activations.ndim == 4:

            A = activations[0]
            G = gradients[0]

        # Handle possible [B,H,W,C]
        elif activations.ndim == 4:

            A = activations[0].permute(
                2, 0, 1
            )

            G = gradients[0].permute(
                2, 0, 1
            )

        else:
            raise RuntimeError(
                f"Unexpected target-layer shape: "
                f"{activations.shape}"
            )

        # Grad-CAM++ weighting approximation
        grad_2 = G.pow(2)
        grad_3 = G.pow(3)

        sum_activations = A.sum(
            dim=(1, 2),
            keepdim=True
        )

        alpha_denom = (
            2 * grad_2
            +
            sum_activations * grad_3
        )

        alpha_denom = torch.where(
            alpha_denom != 0,
            alpha_denom,
            torch.ones_like(alpha_denom)
        )

        alpha = grad_2 / alpha_denom

        positive_gradients = F.relu(G)

        weights = (
            alpha * positive_gradients
        ).sum(
            dim=(1, 2)
        )

        cam = torch.sum(
            weights[:, None, None] * A,
            dim=0
        )

        cam = F.relu(cam)

        cam -= cam.min()

        cam /= (
            cam.max() + 1e-8
        )

        cam = cam.detach().cpu().numpy()

        return cam, target_class


# =========================
# 32. SELECT CONVNEXT TARGET LAYER
# =========================
# The last ConvNeXt stage is used because it retains
# spatial feature information useful for Grad-CAM++.

target_layer = model.convnext.stages[-1]

gradcam = GradCAMPlusPlus(
    model,
    target_layer
)

print("Grad-CAM++ target layer:")
print(target_layer)


# =========================
# 33. GRAD-CAM++ OVERLAY FUNCTION
# =========================
def overlay_heatmap(
    original_image,
    heatmap,
    alpha=0.45
):

    original = original_image.resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    heatmap_img = Image.fromarray(
        np.uint8(255 * heatmap)
    ).resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        Image.Resampling.BILINEAR
    )

    heatmap_array = np.asarray(
        heatmap_img
    )

    cmap = plt.get_cmap("jet")

    colored = cmap(
        heatmap_array / 255.0
    )[:, :, :3]

    colored = np.uint8(
        colored * 255
    )

    colored = Image.fromarray(
        colored
    )

    overlay = Image.blend(
        original,
        colored,
        alpha=alpha
    )

    return overlay


# =========================
# 34. GRAD-CAM++ SINGLE IMAGE
# =========================
sample_path = val_paths[0]

original = Image.open(
    sample_path
).convert("RGB")

enhanced = wavelet_bemd_inspired_enhancement(
    original
)

rgb_tensor = val_tensor_transform(
    original
).unsqueeze(0).to(DEVICE)

enhanced_tensor = val_tensor_transform(
    enhanced
).unsqueeze(0).to(DEVICE)

model.eval()

cam, predicted_class = gradcam.generate(
    rgb_tensor,
    enhanced_tensor
)

overlay = overlay_heatmap(
    original,
    cam
)

true_class = int(
    val_labels[
        np.where(val_paths == sample_path)[0][0]
    ]
)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(original)
plt.title(
    f"Original\nTrue: {CLASS_NAMES[true_class]}"
)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(enhanced)
plt.title("Wavelet + BEMD-inspired")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title(
    f"Grad-CAM++\nPredicted: "
    f"{CLASS_NAMES[predicted_class]}"
)
plt.axis("off")

plt.tight_layout()
plt.show()


# =========================
# 35. SELECT ONE IMAGE PER CLASS
# =========================
selected_samples = []

for class_idx in range(NUM_CLASSES):

    indices = np.where(
        val_labels == class_idx
    )[0]

    if len(indices) == 0:
        continue

    # Prefer a correctly classified sample
    correct_indices = [
        idx for idx in indices
        if y_pred[idx] == class_idx
    ]

    if len(correct_indices) > 0:
        selected_samples.append(
            correct_indices[0]
        )
    else:
        selected_samples.append(
            indices[0]
        )


# =========================
# 36. FIVE-CLASS GRAD-CAM++ FIGURE
# =========================
fig, axes = plt.subplots(
    len(selected_samples),
    3,
    figsize=(12, 4 * len(selected_samples))
)

if len(selected_samples) == 1:
    axes = np.expand_dims(
        axes,
        axis=0
    )

for row, idx in enumerate(
    selected_samples
):

    path = val_paths[idx]

    original = Image.open(
        path
    ).convert("RGB")

    enhanced = wavelet_bemd_inspired_enhancement(
        original
    )

    rgb_tensor = val_tensor_transform(
        original
    ).unsqueeze(0).to(DEVICE)

    enhanced_tensor = val_tensor_transform(
        enhanced
    ).unsqueeze(0).to(DEVICE)

    cam, pred_class = gradcam.generate(
        rgb_tensor,
        enhanced_tensor
    )

    overlay = overlay_heatmap(
        original,
        cam
    )

    true_class = val_labels[idx]

    axes[row, 0].imshow(
        original
    )

    axes[row, 0].set_title(
        f"Original\n"
        f"True: {CLASS_NAMES[true_class]}"
    )

    axes[row, 1].imshow(
        enhanced
    )

    axes[row, 1].set_title(
        "Wavelet + BEMD-inspired"
    )

    axes[row, 2].imshow(
        overlay
    )

    axes[row, 2].set_title(
        f"Grad-CAM++\n"
        f"Predicted: {CLASS_NAMES[pred_class]}"
    )

    for col in range(3):
        axes[row, col].axis("off")

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "GradCAM_PlusPlus_All_Classes.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================
# 37. ATTENTION WEIGHT VISUALIZATION
# =========================
@torch.no_grad()
def get_attention_weights(
    model,
    image_path
):

    original = Image.open(
        image_path
    ).convert("RGB")

    enhanced = wavelet_bemd_inspired_enhancement(
        original
    )

    rgb_tensor = val_tensor_transform(
        original
    ).unsqueeze(0).to(DEVICE)

    enhanced_tensor = val_tensor_transform(
        enhanced
    ).unsqueeze(0).to(DEVICE)

    outputs = model(
        rgb_tensor,
        enhanced_tensor,
        return_features=True
    )

    logits = outputs["logits"]

    prediction = logits.argmax(
        dim=1
    ).item()

    weights = outputs[
        "attention_weights"
    ][0, :, 0].cpu().numpy()

    return (
        original,
        enhanced,
        weights,
        prediction
    )


# =========================
# 38. SAVE METRICS
# =========================
metrics_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1",
        "Macro F1",
        "Balanced Accuracy",
        "MCC",
        "Weighted ROC-AUC"
    ],

    "Value": [
        val_accuracy,
        precision,
        recall,
        f1,
        macro_f1,
        balanced_acc,
        mcc,
        roc_auc
    ]
})

metrics_summary.to_csv(
    os.path.join(OUTPUT_DIR, "ovarian_model_metrics.csv"),
    index=False
)

class_metrics.to_csv(
    os.path.join(OUTPUT_DIR, "ovarian_classwise_metrics.csv"),
    index=False
)

print("\nMetrics saved.")


# =========================
# 39. SAVE CLASSIFICATION REPORT
# =========================
with open(
    os.path.join(OUTPUT_DIR, "classification_report.txt"),
    "w"
) as f:

    f.write(report)

print(
    "Classification report saved."
)


# =========================
# 40. SAVE MODEL
# =========================
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "image_size": IMAGE_SIZE,
        "architecture":
            "Wavelet-BEMD-inspired Swin-ConvNeXt Cross-Attention"
    },
    os.path.join(OUTPUT_DIR, "final_ovarian_swin_convnext.pth")
)

print(
    "\nModel saved to:",
    os.path.join(OUTPUT_DIR, "final_ovarian_swin_convnext.pth")
)


# =========================
# 41. FINAL SUMMARY
# =========================
print("\n" + "=" * 65)
print("FINAL MODEL SUMMARY")
print("=" * 65)

print("Architecture:")
print(
    "Original RGB -> Pretrained Swin Transformer"
)

print(
    "Wavelet/BEMD-enhanced RGB -> Pretrained ConvNeXt"
)

print(
    "Swin + ConvNeXt -> Bidirectional Cross-Attention"
)

print(
    "Cross-Attention -> Attention Pooling -> Dense Classifier"
)

print(
    "Explainability -> Grad-CAM++ on ConvNeXt final stage"
)

print("\nClasses:")
print(", ".join(CLASS_NAMES))

print(
    f"\nBest Validation Accuracy: "
    f"{best_val_acc*100:.2f}%"
)

print(
    "\nReference-paper testing accuracy: 97.37%"
)

print(
    "\nImportant: the new architecture is designed to improve "
    "performance, but accuracy above 97.37% must be demonstrated "
    "experimentally using the same evaluation protocol."
)

print("\nOutput files:")
print(os.path.join(OUTPUT_DIR, "best_ovarian_model.pth"))
print(os.path.join(OUTPUT_DIR, "final_ovarian_swin_convnext.pth"))
print(os.path.join(OUTPUT_DIR, "GradCAM_PlusPlus_All_Classes.png"))
print(os.path.join(OUTPUT_DIR, "ovarian_model_metrics.csv"))
print(os.path.join(OUTPUT_DIR, "ovarian_classwise_metrics.csv"))
print(os.path.join(OUTPUT_DIR, "classification_report.txt"))
